# Entrenamiento del Modelo

## Celda 1 — Librerías

In [1]:
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

## Celda 2 — Configurar MLflow

In [3]:
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("F1 Podium Predictor")

2026/07/09 11:00:07 INFO mlflow.tracking.fluent: Experiment with name 'F1 Podium Predictor' does not exist. Creating a new experiment.


<Experiment: artifact_location='/Users/luisvazquez/Desktop/f1-podium-predictor/notebooks/mlruns/1', creation_time=1783616407707, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1783616407707, lifecycle_stage='active', name='F1 Podium Predictor', tags={}, trace_location=None, workspace='default'>

## Celda 3 — Cargar dataset

In [4]:
dataset = pd.read_csv("../data/processed/f1_dataset.csv")
dataset.head()

,driver,constructor,circuit,year,grid,podium
0,Hamilton,McLaren,albert_park,2008,1,1
1,Heidfeld,BMW Sauber,albert_park,2008,5,1
2,Rosberg,Williams,albert_park,2008,7,1
3,Alonso,Renault,albert_park,2008,11,0
4,Kovalainen,McLaren,albert_park,2008,3,0


## Celda 4 — Separar variables

In [5]:
X = dataset.drop(columns=["podium"])
y = dataset["podium"]

## Celda 5 — Definir columnas

In [6]:
categorical_features = ["driver", "constructor", "circuit"]
numeric_features = ["year", "grid"]

## Celda 6 — Preprocesamiento

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

## Celda 7 — Modelo y pipeline

In [8]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

## Celda 8 — Train/test split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

## Celda 9 — Entrenar con MLflow

In [10]:
with mlflow.start_run(run_name="random_forest_baseline"):

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("class_weight", "balanced")

    mlflow.log_metric("accuracy", accuracy)

    mlflow.sklearn.log_model(
        pipeline,
        name="f1_podium_model"
    )

    print("Accuracy:", accuracy)
    print(classification_report(y_test, predictions))

Accuracy: 0.8636023916292974
              precision    recall  f1-score   support

           0       0.94      0.90      0.92      4673
           1       0.47      0.61      0.53       679

    accuracy                           0.86      5352
   macro avg       0.71      0.75      0.72      5352
weighted avg       0.88      0.86      0.87      5352



## Celda 10 — Guardar modelo para FastAPI

In [11]:
joblib.dump(pipeline, "../models/f1_podium_model.pkl")

['../models/f1_podium_model.pkl']

## Celda 11 — Probar modelo guardado

In [12]:
loaded_model = joblib.load("../models/f1_podium_model.pkl")

sample = pd.DataFrame([{
    "driver": "Hamilton",
    "constructor": "McLaren",
    "circuit": "albert_park",
    "year": 2008,
    "grid": 1
}])

loaded_model.predict(sample), loaded_model.predict_proba(sample)

(array([1]), array([[0., 1.]]))